[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/13_dense_sparse_late_interaction/13_dense_sparse_late_interaction.ipynb)

# Módulo 13 · Dense vs sparse vs late interaction: BM25, SPLADE, ColBERT, MUVERA y retrieval híbrido

**Nivel:** 🟠 Avanzado · **Duración:** ≈ 5 h (lección ≈ 3 h + proyecto ≈ 2 h) · **GPU recomendada:** T4 (todo corre en CPU con `FAST_DEV_RUN = True`) ·
**Cómputo estimado:** 0 unidades en CPU; ≈ 1–2 unidades de Colab en T4 si activas BGE-M3 y el reranker real (estimación orientativa).

> 🧭 **Ruta núcleo vs ampliación.** Núcleo (≈ 3 h): secciones 1–4 (BM25, ColBERT/MaxSim), 7 (fusión) y 8 (rerankers). Ampliación (≈ 1 h más): SPLADE (3), MUVERA y PLAID (5–6) y late chunking (9).

**Prerrequisitos:** módulo 00 (BM25 como baseline), 10 (similitudes, pooling), 11 (contrastivo) y 12 (embedders modernos: el retriever denso de hoy es el de allí).

**Hilo conductor — Políglota.** El CTO te enseña tres consultas reales que fallan: una con un **identificador exacto** (`ERR_CONN_RESET`) que el embedder denso no encuentra, una **parafraseada** (ES→EN) que BM25 no encuentra y una **muy específica** donde el top-10 contiene lo correcto en la posición 7. Este módulo construye el sistema que arregla las tres: **retrieval híbrido (léxico + denso + late interaction) con fusión y reranking**.

> **Estado del arte a octubre de 2026.** Toda cifra, ID de checkpoint o adopción por parte de una empresa va fechada y con fuente; lo que no se pudo verificar se marca como tal.

## 🎯 Objetivos de aprendizaje
Al terminar serás capaz de:
1. **Implementar BM25 desde cero** (índice invertido en matriz dispersa), verificarlo contra `rank_bm25` y explicar el efecto de `k1`, `b` y del *analyzer* (acentos, stemming).
2. **Explicar y calcular** la representación de SPLADE (activación MLM → ReLU → log → max-pooling, regularizador FLOPS) y usar un modelo sparse aprendido.
3. **Implementar MaxSim (ColBERT) desde cero**, vectorizado, y entender su coste de almacenamiento y la compresión residual de ColBERTv2.
4. **Implementar una versión mini de PLAID** (interacción por centroides + filtrado) y medir el *trade-off* candidatos/recall.
5. **Implementar MUVERA** (Fixed Dimensional Encodings) y medir cuánto se parece su producto interno a MaxSim.
6. **Fusionar retrievers** con RRF y con combinación lineal normalizada, y decidir cuándo usar cada una.
7. **Montar el pipeline recuperar → fusionar → reordenar** con un cross-encoder y medir el coste/beneficio por profundidad de reranking.
8. **Aplicar late chunking** y razonar cuándo ayuda.

## 🔁 Conexión con módulos anteriores
<details><summary><b>1 · (M00)</b> ¿Qué fórmula usa BM25 y qué problema de TF-IDF resuelve la saturación de <code>k1</code>?</summary>

$\text{BM25}(q,d)=\sum_{t\in q}\text{IDF}(t)\,\frac{f_{t,d}(k_1+1)}{f_{t,d}+k_1\,(1-b+b\,|d|/\overline{|d|})}$. En TF-IDF el peso crece linealmente con la frecuencia del término; BM25 lo **satura** (un término que aparece 20 veces no vale 20× más) y normaliza por longitud.
</details>
<details><summary><b>2 · (M10/M12)</b> ¿Por qué un embedder denso (un vector por texto) pierde información sobre términos raros o identificadores exactos?</summary>

Porque comprime todo el texto en $d$ números: la señal de un token raro (un código de error, un nombre propio) se diluye en el promedio y el modelo apenas vio esos tokens al entrenar. Un índice invertido, en cambio, hace match **exacto** de ese término.
</details>
<details><summary><b>3 · (M04/M05)</b> ¿Cómo influye la tokenización en un retriever léxico?</summary>

El *analyzer* decide qué es un término: minúsculas, tildes, stemming, separación de identificadores (`snake_case`, `CamelCase`). Con otra tokenización el mismo corpus produce otro índice y otras métricas. Es el primer lugar donde mirar cuando BM25 "no encuentra" algo.
</details>
<details><summary><b>4 · (M11/M12)</b> ¿Qué distingue a un <i>bi-encoder</i> de un <i>cross-encoder</i> en coste y calidad?</summary>

El bi-encoder codifica query y documento por separado (los documentos se precomputan): $O(1)$ forward por búsqueda sobre el índice, menor calidad. El cross-encoder procesa el par conjuntamente: más preciso, pero exige un forward **por candidato** y no puede precomputarse.
</details>

## 🗺️ Mapa de madurez (a oct-2026)
🟢 sólido · 🟡 emergente · 🔴 experimental.

| Técnica | Madurez | Quién la usa (respaldado) | Fuente |
|---|---|---|---|
| BM25 / índice invertido | 🟢 | motores de búsqueda léxicos (Lucene, Elasticsearch, OpenSearch, Vespa…) | Robertson & Zaragoza 2009 |
| Sparse aprendido (SPLADE, SPLADE-v3) | 🟢 inglés · 🟡 multilingüe | Elastic (ELSER) y OpenSearch (neural sparse) ofrecen modelos sparse aprendidos | Formal et al. 2021–2024; docs de Elastic/OpenSearch |
| BGE-M3 (denso + sparse + multi-vector en un modelo) | 🟢 | BAAI; integrado en el ecosistema de FlagEmbedding | Chen et al. 2024 (2402.03216) |
| Late interaction ColBERT / ColBERTv2 | 🟡 | Vespa (embedder ColBERT nativo), Qdrant (multivector `MAX_SIM`), Jina-ColBERT-v2 | Khattab & Zaharia 2020; Santhanam et al. 2021; blogs/docs |
| PLAID / WARP (motores de late interaction) | 🟡 | implementación de referencia de ColBERT (PLAID); WARP es investigación 2025 | 2205.09707; 2501.17788 |
| MUVERA (multi-vector → un vector de dimensión fija) | 🟡 | Weaviate documenta una opción de codificación MUVERA | Dhulipala et al. 2024 (2405.19504); docs de Weaviate |
| Fusión RRF / combinación convexa | 🟢 | práctica estándar de búsqueda híbrida | Cormack et al. 2009; Bruch et al. 2022 |
| Rerankers cross-encoder | 🟢 | BAAI (bge-reranker-v2-m3), muchos proveedores | Nogueira & Cho 2019 (1901.04085) |
| Rerankers basados en LLM (Qwen3-Reranker, bge-reranker-v2-gemma) | 🟡 | Alibaba, BAAI — mayor coste | 2506.05176; tarjetas |
| Late chunking | 🟡 | Jina AI | Günther et al. 2024 (2409.04701) |
| Búsqueda híbrida (léxico + denso) como configuración por defecto | 🟢 | prácticamente todas las bases vectoriales/motores modernos | docs de los motores |

## 0 · Preparación
Como en el módulo 12: celdas de utilidades autocontenidas (corpus canónico del módulo 00, métricas, embedder con fallback). Usaremos **dos bancos de pruebas** para ver *dónde gana cada retriever*:

- **Canónico (Políglota):** el corpus del módulo 00 (Wikipedia ES/EN + código si hay acceso; si no, sintético con un `⚠️`). Queries con bastante solapamiento léxico.
- **Paráfrasis ES↔EN↔código:** 29 conceptos técnicos × 3 documentos con **58 queries parafraseadas** que casi no comparten palabras con su documento (y buscan entre idiomas). Pequeño y escrito a mano: sirve para *diagnosticar*, no para reportar cifras.

In [ ]:
!pip install -q -U sentence-transformers transformers scikit-learn scipy matplotlib pandas rank_bm25

In [ ]:
import os, sys, time, math, random, json, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")

import numpy as np

SEED = 42
random.seed(SEED); np.random.seed(SEED)
FAST_DEV_RUN = True      # True: tamaños diminutos (CPU en minutos). Pon False en Colab con GPU para la escala completa.
N_THREADS = int(os.environ.get("N_THREADS", "1" if FAST_DEV_RUN else "4"))
os.environ.setdefault("OMP_NUM_THREADS", str(N_THREADS))

try:
    import torch
    torch.manual_seed(SEED)
    torch.set_num_threads(N_THREADS)
    device = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:      # los fragmentos "desde cero" de NumPy funcionan sin torch
    torch, device = None, "cpu"
print(f"device={device} · FAST_DEV_RUN={FAST_DEV_RUN} · threads={N_THREADS}")

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

In [ ]:
# ── Métricas de retrieval canónicas de Políglota (módulo 00) ──────────────────
import math

import numpy as np


def _dedup(ranked_ids):
    """Quita duplicados conservando la primera aparición (un doc repetido NO puntúa dos veces)."""
    seen, out = set(), []
    for d in ranked_ids:
        if d not in seen:
            seen.add(d)
            out.append(d)
    return out


def dcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """DCG con ganancia lineal (como trec_eval / pytrec_eval, que usa BEIR/MTEB) y descuento 1/log2(rango+1)."""
    ranked = _dedup(ranked_ids)[:k]
    return sum(rels.get(d, 0) / math.log2(i + 2) for i, d in enumerate(ranked) if rels.get(d, 0) > 0)


def ndcg_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """nDCG@k. rels: doc_id -> relevancia (>0 relevante). Sin relevantes -> 0.0 (evaluate() las excluye)."""
    ideal = sorted((r for r in rels.values() if r > 0), reverse=True)[:k]
    idcg = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
    return 0.0 if idcg == 0 else dcg_at_k(ranked_ids, rels, k) / idcg


def recall_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """Fracción de los documentos relevantes que aparecen en el top-k."""
    relevant = {d for d, r in rels.items() if r > 0}
    if not relevant:
        return 0.0
    return len(relevant & set(_dedup(ranked_ids)[:k])) / len(relevant)


def mrr_at_k(ranked_ids, rels: dict, k: int = 10) -> float:
    """1/rango del primer relevante dentro del top-k (0 si no hay ninguno)."""
    for i, d in enumerate(_dedup(ranked_ids)[:k]):
        if rels.get(d, 0) > 0:
            return 1.0 / (i + 1)
    return 0.0


def rank_from_scores(scores, doc_ids, k: int | None = None):
    """Ordena por score desc. Empates -> orden original del corpus (determinista). Los empates son una fuente
    silenciosa de varianza: con scores iguales, el orden de entrada decidiría el resultado."""
    scores = np.asarray(scores, dtype=float)
    order = np.lexsort((np.arange(len(scores)), -scores))
    order = order[:k] if k else order
    return [doc_ids[i] for i in order]


def evaluate(run: dict, qrels: dict, k: int = 10) -> dict:
    """run: {qid: [doc_id ordenados]}. Promedia solo sobre queries con algún relevante (convención trec_eval)."""
    n, tot = 0, {"ndcg": 0.0, "recall": 0.0, "mrr": 0.0}
    for qid, rels in qrels.items():
        if not any(r > 0 for r in rels.values()):
            continue
        ranked = run.get(qid, [])
        tot["ndcg"] += ndcg_at_k(ranked, rels, k)
        tot["recall"] += recall_at_k(ranked, rels, k)
        tot["mrr"] += mrr_at_k(ranked, rels, k)
        n += 1
    out = {m: (v / n if n else 0.0) for m, v in tot.items()}
    out.update(n_queries=n, n_skipped=len(qrels) - n)
    return out

In [ ]:
def ndcgs_por_query(run: dict, qrels: dict, k: int = 10):
    """nDCG@k de cada query (mismo filtro que evaluate(): solo queries con algún relevante). Sirve para bootstrap."""
    return np.array([ndcg_at_k(run.get(q, []), r, k) for q, r in qrels.items() if any(v > 0 for v in r.values())])

In [ ]:
# Mini-corpus sintético "Políglota": 29 conceptos técnicos x {EN, ES, código}. Es el FALLBACK del cargador de datos;
# tiene el mismo esquema que el corpus real. Las queries están parafraseadas a propósito para que no compartan
# casi ninguna palabra con el documento (así BM25 no es trivial y los embeddings multilingües sí aportan).
PARAFRASIS = [
 ("hnsw",
  "HNSW builds a multi-layer proximity graph. A search enters at the top layer, walks greedily towards the target and descends layer by layer, returning approximate neighbors in logarithmic time.",
  "HNSW construye un grafo de proximidad con varias capas. La búsqueda entra por la capa superior, avanza de forma voraz hacia el objetivo y desciende capa a capa, devolviendo vecinos aproximados en tiempo logarítmico.",
  "def greedy_search(graph, vectors, q, entry):\n    best, improved = entry, True\n    while improved:\n        improved = False\n        for nb in graph[best]:\n            if dist(vectors[nb], q) < dist(vectors[best], q):\n                best, improved = nb, True\n    return best",
  "find similar vectors fast by hopping through a layered network", "buscar vectores parecidos rápido saltando por una red de niveles"),
 ("inverted_index",
  "An inverted index maps every term to the list of documents that contain it, so a keyword query only reads the posting lists of its terms instead of scanning the whole collection.",
  "Un índice invertido asocia cada término con la lista de documentos que lo contienen, de modo que una consulta solo lee las listas de sus términos en lugar de recorrer toda la colección.",
  "def build_index(docs):\n    index = {}\n    for doc_id, text in docs.items():\n        for term in set(text.lower().split()):\n            index.setdefault(term, []).append(doc_id)\n    return index",
  "data structure that avoids scanning every document for a word", "estructura que evita recorrer todos los textos al buscar una palabra"),
 ("bm25",
  "BM25 scores a document by summing, for each query term, its inverse document frequency times a saturating term frequency, normalized by document length with parameters k1 and b.",
  "BM25 puntúa un documento sumando, para cada término de la consulta, su frecuencia inversa de documento por una frecuencia de término saturada, normalizada por la longitud del documento con los parámetros k1 y b.",
  "def bm25_term(tf, idf, dl, avgdl, k1=1.2, b=0.75):\n    norm = k1 * (1 - b + b * dl / avgdl)\n    return idf * tf * (k1 + 1) / (tf + norm)",
  "classic keyword ranking formula with length normalization", "fórmula clásica de ranking por palabras clave que penaliza textos largos"),
 ("bpe",
  "Byte-pair encoding starts from single bytes and repeatedly merges the most frequent adjacent pair into a new token until the vocabulary reaches the desired size.",
  "La codificación por pares de bytes parte de bytes sueltos y fusiona repetidamente el par adyacente más frecuente en un nuevo token hasta alcanzar el tamaño de vocabulario deseado.",
  "def most_frequent_pair(ids):\n    counts = {}\n    for a, b in zip(ids, ids[1:]):\n        counts[(a, b)] = counts.get((a, b), 0) + 1\n    return max(counts, key=counts.get)",
  "how a language model splits text into subword pieces by merging common pairs", "cómo un modelo de lenguaje trocea el texto en piezas fusionando pares comunes"),
 ("cosine",
  "Cosine similarity measures the angle between two vectors: the dot product divided by the product of their norms. It ignores magnitude and ranges from -1 to 1.",
  "La similitud del coseno mide el ángulo entre dos vectores: el producto punto dividido por el producto de sus normas. Ignora la magnitud y va de -1 a 1.",
  "def cosine(a, b):\n    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))",
  "compare two vectors by direction rather than length", "comparar dos vectores por su dirección y no por su longitud"),
 ("kmeans",
  "K-means alternates two steps: assign every point to its nearest centroid, then move each centroid to the mean of its assigned points, until assignments stop changing.",
  "K-means alterna dos pasos: asignar cada punto a su centroide más cercano y mover cada centroide a la media de sus puntos, hasta que las asignaciones dejan de cambiar.",
  "def kmeans_step(X, C):\n    d = ((X[:, None] - C[None]) ** 2).sum(-1)\n    a = d.argmin(1)\n    return np.stack([X[a == k].mean(0) for k in range(len(C))]), a",
  "group unlabeled points into clusters around moving centers", "agrupar puntos sin etiquetas alrededor de centros que se desplazan"),
 ("pq",
  "Product quantization splits a vector into sub-vectors and replaces each by the index of its nearest codeword, compressing a float vector to a few bytes while allowing fast approximate distances.",
  "La cuantización de producto divide un vector en subvectores y sustituye cada uno por el índice de su palabra de código más cercana, comprimiendo un vector de floats a pocos bytes con distancias aproximadas rápidas.",
  "def pq_encode(x, codebooks):\n    subs = np.split(x, len(codebooks))\n    return [int(((cb - s) ** 2).sum(1).argmin()) for s, cb in zip(subs, codebooks)]",
  "shrink embeddings to a handful of bytes using small codebooks", "reducir embeddings a unos pocos bytes usando diccionarios pequeños"),
 ("reranker",
  "A cross-encoder reranker reads the query and a candidate document together in one transformer pass and outputs a relevance score; it is accurate but too slow for the whole corpus.",
  "Un reranker cross-encoder lee la consulta y un documento candidato juntos en una sola pasada del transformer y devuelve una puntuación de relevancia; es preciso pero demasiado lento para todo el corpus.",
  "def rerank(model, query, candidates, top_n=10):\n    scores = model.predict([(query, c) for c in candidates])\n    order = np.argsort(-scores)[:top_n]\n    return [candidates[i] for i in order]",
  "second-stage model that rescores a short list of candidates jointly with the question", "modelo de segunda etapa que reordena una lista corta mirando pregunta y texto a la vez"),
 ("matryoshka",
  "Matryoshka representation learning trains an embedding so that its first m dimensions already form a good smaller embedding, letting you truncate vectors and re-normalize them to save memory.",
  "El aprendizaje de representaciones Matryoshka entrena un embedding para que sus primeras m dimensiones ya formen un embedding más pequeño útil, permitiendo truncar los vectores y renormalizarlos para ahorrar memoria.",
  "def truncate(emb, m):\n    out = emb[:, :m]\n    return out / np.linalg.norm(out, axis=1, keepdims=True)",
  "cut a vector to fewer coordinates without retraining", "recortar un vector a menos coordenadas sin volver a entrenar"),
 ("contrastive",
  "Contrastive loss pulls the embeddings of matching pairs together and pushes apart the other examples in the batch, using a softmax over similarities scaled by a temperature.",
  "La pérdida contrastiva acerca los embeddings de los pares que coinciden y aleja el resto de ejemplos del lote, usando un softmax sobre similitudes escaladas por una temperatura.",
  "def info_nce(q, d, tau=0.05):\n    logits = (q @ d.T) / tau\n    labels = torch.arange(len(q))\n    return torch.nn.functional.cross_entropy(logits, labels)",
  "training objective that treats other items in the batch as wrong answers", "objetivo de entrenamiento que trata los demás ejemplos del lote como respuestas erróneas"),
 ("attention",
  "Self-attention lets every token build its representation as a weighted average of all value vectors, with weights given by the softmax of scaled query-key dot products.",
  "La autoatención permite que cada token construya su representación como un promedio ponderado de todos los vectores de valor, con pesos dados por el softmax de productos punto consulta-clave escalados.",
  "def attention(q, k, v):\n    w = torch.softmax(q @ k.T / q.shape[-1] ** 0.5, dim=-1)\n    return w @ v",
  "mechanism where each word looks at all the other words to update itself", "mecanismo en el que cada palabra mira a todas las demás para actualizarse"),
 ("cosine_decay",
  "A cosine learning-rate schedule starts at the peak value after warmup and decays smoothly towards zero following half a cosine wave, avoiding abrupt drops late in training.",
  "Un calendario de tasa de aprendizaje con coseno parte del valor máximo tras el calentamiento y decae suavemente hacia cero siguiendo media onda de coseno, evitando caídas bruscas al final del entrenamiento.",
  "def lr_at(step, total, peak=3e-4):\n    return 0.5 * peak * (1 + math.cos(math.pi * step / total))",
  "how to shrink the step size smoothly while a network trains", "cómo reducir poco a poco el tamaño del paso mientras una red entrena"),
 ("dropout",
  "Dropout randomly zeroes a fraction of activations during training and rescales the rest, which discourages co-adaptation of neurons and acts as a regularizer; it is disabled at inference.",
  "Dropout pone a cero al azar una fracción de las activaciones durante el entrenamiento y reescala el resto, lo que evita la coadaptación de neuronas y actúa como regularizador; se desactiva en inferencia.",
  "def dropout(x, p=0.1, training=True):\n    if not training:\n        return x\n    mask = (np.random.rand(*x.shape) > p) / (1 - p)\n    return x * mask",
  "randomly switching off neurons to reduce overfitting", "apagar neuronas al azar para reducir el sobreajuste"),
 ("layernorm",
  "Layer normalization standardizes each token's feature vector to zero mean and unit variance and then applies a learned scale and shift, stabilizing the training of deep transformers.",
  "La normalización por capas estandariza el vector de características de cada token a media cero y varianza uno y luego aplica una escala y un desplazamiento aprendidos, estabilizando el entrenamiento de transformers profundos.",
  "def layer_norm(x, g, b, eps=1e-5):\n    mu = x.mean(-1, keepdims=True)\n    var = x.var(-1, keepdims=True)\n    return g * (x - mu) / np.sqrt(var + eps) + b",
  "rescale each feature vector so deep networks train stably", "reescalar cada vector de características para que las redes profundas entrenen estables"),
 ("grad_clip",
  "Gradient clipping rescales the gradient whenever its global norm exceeds a threshold, which prevents rare huge updates from destabilizing training.",
  "El recorte de gradiente reescala el gradiente cada vez que su norma global supera un umbral, lo que evita que actualizaciones enormes y poco frecuentes desestabilicen el entrenamiento.",
  "def clip_grad(grads, max_norm=1.0):\n    total = math.sqrt(sum(float((g ** 2).sum()) for g in grads))\n    scale = min(1.0, max_norm / (total + 1e-6))\n    return [g * scale for g in grads]",
  "limit the size of parameter updates to avoid exploding values", "limitar el tamaño de las actualizaciones de parámetros para evitar valores que explotan"),
 ("lru_cache",
  "An LRU cache keeps the most recently used entries and evicts the least recently used one when it is full, giving constant-time lookups with a hash map plus a linked list.",
  "Una caché LRU conserva las entradas usadas más recientemente y expulsa la menos reciente cuando se llena, con búsquedas en tiempo constante mediante un mapa hash y una lista enlazada.",
  "from collections import OrderedDict\n\nclass LRU:\n    def __init__(self, cap):\n        self.cap, self.d = cap, OrderedDict()\n    def get(self, k):\n        self.d.move_to_end(k)\n        return self.d[k]",
  "memory that forgets whatever was touched longest ago", "memoria que olvida lo que se tocó hace más tiempo"),
 ("token_bucket",
  "A token bucket rate limiter adds tokens at a fixed rate up to a maximum capacity; each request consumes one token and is rejected when the bucket is empty, allowing short bursts.",
  "Un limitador de tasa de cubeta de tokens añade tokens a ritmo fijo hasta una capacidad máxima; cada petición consume un token y se rechaza si la cubeta está vacía, lo que permite ráfagas cortas.",
  "def allow(bucket, now, rate=5.0, cap=10.0):\n    bucket['tokens'] = min(cap, bucket['tokens'] + (now - bucket['t']) * rate)\n    bucket['t'] = now\n    if bucket['tokens'] >= 1:\n        bucket['tokens'] -= 1\n        return True\n    return False",
  "stop an API client from sending too many requests per second", "impedir que un cliente de una API envíe demasiadas peticiones por segundo"),
 ("binary_search",
  "Binary search finds a value in a sorted array by repeatedly comparing it with the middle element and discarding half of the remaining range, so it needs only logarithmic comparisons.",
  "La búsqueda binaria encuentra un valor en un arreglo ordenado comparándolo repetidamente con el elemento central y descartando la mitad del rango restante, por lo que solo necesita comparaciones logarítmicas.",
  "def binary_search(a, x):\n    lo, hi = 0, len(a) - 1\n    while lo <= hi:\n        mid = (lo + hi) // 2\n        if a[mid] == x:\n            return mid\n        lo, hi = (mid + 1, hi) if a[mid] < x else (lo, mid - 1)\n    return -1",
  "locate an item in an ordered list by halving the range each time", "localizar un elemento en una lista ordenada partiendo el rango a la mitad cada vez"),
 ("hash_collision",
  "A hash table maps keys to buckets with a hash function; when two keys land in the same bucket a collision occurs, resolved by chaining or open addressing.",
  "Una tabla hash asigna claves a cubetas mediante una función hash; cuando dos claves caen en la misma cubeta ocurre una colisión, que se resuelve con encadenamiento o direccionamiento abierto.",
  "def put(table, key, value):\n    bucket = table[hash(key) % len(table)]\n    for i, (k, _) in enumerate(bucket):\n        if k == key:\n            bucket[i] = (key, value)\n            return\n    bucket.append((key, value))",
  "what happens when two different keys map to the same slot", "qué pasa cuando dos claves distintas caen en la misma casilla"),
 ("deadlock",
  "A deadlock happens when two threads each hold a lock the other needs and wait forever; acquiring locks in a single global order prevents the circular wait.",
  "Un interbloqueo ocurre cuando dos hilos retienen cada uno un candado que el otro necesita y esperan para siempre; adquirir los candados en un único orden global evita la espera circular.",
  "def transfer(a, b, amount):\n    first, second = sorted([a, b], key=id)\n    with first.lock:\n        with second.lock:\n            a.balance -= amount\n            b.balance += amount",
  "two threads waiting on each other forever", "dos hilos esperándose mutuamente para siempre"),
 ("btree_index",
  "A B-tree index keeps keys sorted in a balanced tree with wide nodes, so a database can find a row or scan a range with only a few disk reads.",
  "Un índice B-tree mantiene las claves ordenadas en un árbol balanceado con nodos anchos, de modo que una base de datos puede encontrar una fila o recorrer un rango con pocas lecturas de disco.",
  "def create_index(cursor):\n    cursor.execute('CREATE INDEX idx_users_email ON users (email)')\n    cursor.execute('EXPLAIN SELECT * FROM users WHERE email = ?', ('a@b.c',))",
  "make database lookups on a column faster with a sorted tree", "acelerar las búsquedas de una columna en una base de datos con un árbol ordenado"),
 ("sharding",
  "Sharding splits a large dataset across several machines by a shard key, so each node stores and serves only part of the data and capacity grows horizontally.",
  "El sharding reparte un conjunto de datos grande entre varias máquinas según una clave de partición, de modo que cada nodo almacena y sirve solo una parte y la capacidad crece horizontalmente.",
  "def shard_for(key, n_shards):\n    h = int(hashlib.md5(key.encode()).hexdigest(), 16)\n    return h % n_shards",
  "distribute data over many servers so no single one holds everything", "distribuir los datos entre muchos servidores para que ninguno guarde todo"),
 ("backoff",
  "Exponential backoff retries a failed request after waiting 2^n times a base delay, usually with random jitter, so that many clients do not hammer a recovering service at once.",
  "El reintento con retroceso exponencial repite una petición fallida tras esperar 2^n veces un retardo base, normalmente con jitter aleatorio, para que muchos clientes no saturen a la vez un servicio que se recupera.",
  "def retry(fn, tries=5, base=0.5):\n    for n in range(tries):\n        try:\n            return fn()\n        except Exception:\n            time.sleep(base * 2 ** n * random.uniform(0.5, 1.5))\n    raise RuntimeError('agotados los reintentos')",
  "wait longer after each failed network call before trying again", "esperar más tras cada llamada de red fallida antes de volver a intentar"),
 ("mocking",
  "In unit tests a mock replaces a real dependency such as a network client, so the test is fast, deterministic and verifies how the code under test calls it.",
  "En las pruebas unitarias un mock sustituye una dependencia real, como un cliente de red, para que la prueba sea rápida, determinista y verifique cómo la llama el código bajo prueba.",
  "from unittest.mock import patch\n\ndef test_fetch():\n    with patch('app.http_get', return_value={'ok': True}) as m:\n        assert fetch('/x') == {'ok': True}\n        m.assert_called_once_with('/x')",
  "fake a service in a test so it runs without the internet", "simular un servicio en una prueba para que corra sin internet"),
 ("git_rebase",
  "Git rebase replays the commits of your branch on top of another base commit, producing a linear history instead of a merge commit, but it rewrites commit hashes.",
  "Git rebase reaplica los commits de tu rama encima de otro commit base, produciendo un historial lineal en lugar de un commit de fusión, pero reescribe los hashes de los commits.",
  "import subprocess\n\ndef rebase_onto_main():\n    subprocess.run(['git', 'fetch', 'origin'], check=True)\n    subprocess.run(['git', 'rebase', 'origin/main'], check=True)",
  "move my branch commits to the tip of main to keep a straight history", "mover los commits de mi rama a la punta de main para mantener un historial recto"),
 ("docker",
  "A Docker container packages an application with its dependencies and runs it as an isolated process that shares the host kernel, which makes deployments reproducible and lightweight.",
  "Un contenedor Docker empaqueta una aplicación con sus dependencias y la ejecuta como un proceso aislado que comparte el kernel del anfitrión, lo que hace los despliegues reproducibles y ligeros.",
  "import subprocess\n\ndef run_container(image, port=8000):\n    subprocess.run(['docker', 'run', '--rm', '-p', f'{port}:{port}', image], check=True)",
  "ship software with everything it needs so it runs the same everywhere", "entregar software con todo lo que necesita para que corra igual en cualquier sitio"),
 ("pagination",
  "Cursor-based pagination returns a page of results plus an opaque cursor pointing to the last item, which is more stable and efficient than offsets when the data changes between requests.",
  "La paginación por cursor devuelve una página de resultados junto con un cursor opaco que apunta al último elemento, algo más estable y eficiente que los desplazamientos cuando los datos cambian entre peticiones.",
  "def list_items(db, cursor=None, limit=20):\n    rows = db.query('SELECT * FROM items WHERE id > ? ORDER BY id LIMIT ?', cursor or 0, limit)\n    return rows, (rows[-1].id if rows else None)",
  "return long API results in chunks that stay consistent while data changes", "devolver resultados largos de una API en trozos que se mantienen coherentes mientras los datos cambian"),
 ("regex",
  "A regular expression describes a set of strings with literals, character classes and quantifiers; the engine scans the text and returns the matches, but nested quantifiers can cause catastrophic backtracking.",
  "Una expresión regular describe un conjunto de cadenas con literales, clases de caracteres y cuantificadores; el motor recorre el texto y devuelve las coincidencias, pero los cuantificadores anidados pueden causar retroceso catastrófico.",
  "import re\n\nEMAIL = re.compile(r'[\\w.+-]+@[\\w-]+\\.[\\w.]+')\n\ndef find_emails(text):\n    return EMAIL.findall(text)",
  "pattern language to extract pieces of text such as e-mail addresses", "lenguaje de patrones para extraer fragmentos de texto como direcciones de correo"),
 ("unicode_norm",
  "Unicode normalization converts equivalent character sequences to a canonical form: NFC composes accents into single code points, while NFD decomposes them into base letter plus combining mark.",
  "La normalización Unicode convierte secuencias de caracteres equivalentes a una forma canónica: NFC compone los acentos en un solo punto de código, mientras que NFD los descompone en letra base más marca combinante.",
  "import unicodedata\n\ndef strip_accents(s):\n    nfd = unicodedata.normalize('NFD', s)\n    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')",
  "make visually identical accented strings compare as equal", "lograr que cadenas con tildes visualmente idénticas se comparen como iguales"),
]
print(f"{len(PARAFRASIS)} conceptos x 3 documentos (en/es/código) en el mini golden set de paráfrasis")

In [ ]:
def corpus_parafrasis() -> dict:
    """Mini golden set de PARÁFRASIS y búsqueda CRUZADA ES<->EN<->código (29 conceptos x 3 documentos, 58 queries).
    Mismo esquema que carga_corpus(). Las queries casi no comparten palabras con sus documentos: BM25 sufre y los
    modelos semánticos/multilingües deberían ganar. Es un conjunto pequeño y escrito a mano: úsalo para diagnosticar,
    no para reportar cifras de producción."""
    docs, queries, qrels = [], [], {}
    for key, en, es, code, q_en, q_es in PARAFRASIS:
        ids = {"en": f"{key}-en", "es": f"{key}-es", "code": f"{key}-code"}
        docs += [{"id": ids["en"], "lang": "en", "text": en}, {"id": ids["es"], "lang": "es", "text": es},
                 {"id": ids["code"], "lang": "code", "text": code}]
        for lang, q in (("en", q_en), ("es", q_es)):
            qid = f"par-{key}-{lang}"
            queries.append({"qid": qid, "lang": lang, "text": q})
            qrels[qid] = {d: 1 for d in ids.values()}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": "paraphrase-golden"}

In [ ]:
# Catálogo de embedders multilingües "pequeños" (caben en CPU/T4). Prefijos copiados de las tarjetas de modelo:
# VERIFÍCALOS en la tarjeta antes de usar un modelo en producción (cambian entre versiones).
MODELOS = {
    "multilingual-e5-small": dict(hf="intfloat/multilingual-e5-small", q="query: ", d="passage: "),
    "multilingual-e5-base": dict(hf="intfloat/multilingual-e5-base", q="query: ", d="passage: "),
    "paraphrase-multilingual-MiniLM-L12-v2": dict(hf="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", q="", d=""),
    "gte-multilingual-base": dict(hf="Alibaba-NLP/gte-multilingual-base", q="", d="", trust=True),
    "bge-m3": dict(hf="BAAI/bge-m3", q="", d=""),
    "nomic-embed-text-v2-moe": dict(hf="nomic-ai/nomic-embed-text-v2-moe", q="search_query: ", d="search_document: ", trust=True),
    "Qwen3-Embedding-0.6B": dict(hf="Qwen/Qwen3-Embedding-0.6B", d="",
                                 q="Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery:"),
    "embeddinggemma-300m": dict(hf="google/embeddinggemma-300m", q="task: search result | query: ", d="title: none | text: ", gated=True),
}

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

class HashSVDEmbedder:
    """Fallback SIN descargas: TF-IDF de n-gramas de caracteres + SVD. NO es multilingüe ni semántico:
    sirve para ejercitar el pipeline (truncado, cuantización, métricas) cuando HF no es accesible."""
    def __init__(self, dim: int = 64, ngram=(3, 5), analyzer: str = "char_wb"):
        self.dim = dim
        self.tfidf = TfidfVectorizer(analyzer=analyzer, ngram_range=ngram, sublinear_tf=True, min_df=1)
        self.svd = TruncatedSVD(n_components=dim, random_state=SEED)
    def fit(self, textos):
        X = self.tfidf.fit_transform(textos)
        self.svd.n_components = min(self.dim, X.shape[1] - 1, X.shape[0] - 1)
        self.svd.fit(X)
        return self
    def encode(self, textos):
        Z = self.svd.transform(self.tfidf.transform(textos))
        return (Z / np.maximum(np.linalg.norm(Z, axis=1, keepdims=True), 1e-9)).astype(np.float32)

class Embedder:
    """Interfaz común: encode_queries / encode_docs (aplican el prefijo de cada modelo) -> float32 normalizado."""
    def __init__(self, clave, cfg, model=None, fallback=None):
        self.clave, self.cfg, self.model, self.fallback = clave, cfg, model, fallback
        self.is_real = model is not None
    def _enc(self, textos, prefijo, bs=16):
        if self.model is not None:
            E = self.model.encode([prefijo + t for t in textos], batch_size=bs, normalize_embeddings=True,
                                  convert_to_numpy=True, show_progress_bar=False)
            return E.astype(np.float32)
        return self.fallback.encode([prefijo + t for t in textos])
    def encode_queries(self, textos): return self._enc(textos, self.cfg.get("q", ""))
    def encode_docs(self, textos): return self._enc(textos, self.cfg.get("d", ""))
    @property
    def n_params(self):
        return sum(p.numel() for p in self.model.parameters()) if self.model is not None else 0

def cargar_embedder(clave, textos_ajuste=None, max_len=256):
    """Carga con sentence-transformers; si no se puede (sin red, modelo con licencia, falta de RAM) cae al fallback."""
    cfg = MODELOS[clave]
    try:
        from sentence_transformers import SentenceTransformer
        m = SentenceTransformer(cfg["hf"], device=device, trust_remote_code=cfg.get("trust", False))
        m.max_seq_length = min(m.max_seq_length or max_len, max_len)
        return Embedder(clave, cfg, model=m)
    except Exception as e:
        print(f"[embedder] {clave}: no se pudo cargar ({type(e).__name__}: {str(e)[:70]})")
        if textos_ajuste is None:
            return None
        print(f"[embedder] {clave}: usando fallback TF-IDF+SVD (NO representa al modelo real)")
        return Embedder(clave + " (fallback)", cfg, fallback=HashSVDEmbedder().fit(textos_ajuste))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

class Bench:
    """Un corpus + sus queries + las matrices de scores (n_queries x n_docs) de cada retriever."""
    def __init__(self, nombre, corpus):
        self.nombre, self.corpus = nombre, corpus
        self.ids = [d["id"] for d in corpus["docs"]]
        self.txt = [d["text"] for d in corpus["docs"]]
        self.qs = corpus["queries"]
        self.qtxt = [q["text"] for q in self.qs]
        self.qrels = corpus["qrels"]
        self.S = {}                                   # nombre del retriever -> matriz de scores
    def run(self, S, k=10):
        return {q["qid"]: rank_from_scores(S[i], self.ids, k) for i, q in enumerate(self.qs)}
    def ndcg(self, S, k=10):
        return evaluate(self.run(S, k), self.qrels, k)["ndcg"]
    def resumen(self):
        filas = []
        for nombre, S in self.S.items():
            r = evaluate(self.run(S), self.qrels)
            filas.append({"retriever": nombre, "nDCG@10": round(r["ndcg"], 3), "Recall@10": round(r["recall"], 3), "MRR@10": round(r["mrr"], 3)})
        return pd.DataFrame(filas).set_index("retriever")

CORPUS = carga_corpus(FAST_DEV_RUN)
PAR = corpus_parafrasis()
B1, B2 = Bench("canónico (Políglota)", CORPUS), Bench("paráfrasis ES↔EN↔código", PAR)
BENCHES = [B1, B2]
for b in BENCHES:
    print(f"{b.nombre}: fuente={b.corpus['source']} · {len(b.ids)} docs · {len(b.qs)} queries")

In [ ]:
# Retriever denso: el primer embedder que se pueda cargar de verdad; si no, el fallback TF-IDF+SVD (módulo 12).
TEXTOS_AJUSTE = [t for b in BENCHES for t in b.txt]
emb = None
for clave in ("multilingual-e5-small", "paraphrase-multilingual-MiniLM-L12-v2"):
    emb = cargar_embedder(clave)
    if emb is not None:
        break
if emb is None:
    emb = cargar_embedder("multilingual-e5-small", textos_ajuste=TEXTOS_AJUSTE)
for b in BENCHES:
    Ed, Eq = emb.encode_docs(b.txt), emb.encode_queries(b.qtxt)
    b.S["denso"] = Eq @ Ed.T
print(f"denso = {emb.clave} · real={emb.is_real}")

## 1 · Intuición: tres maneras de representar un documento

| | **Sparse léxico / aprendido** | **Denso (un vector)** | **Late interaction (multi-vector)** |
|---|---|---|---|
| Representación | vector de $|V|\approx 10^5$ dimensiones, casi todo ceros | 1 vector de $d\approx 10^3$ | 1 vector **por token** ($n\times d$, $d\approx 128$) |
| Índice | **invertido** (término → documentos) | ANN (módulo 14) | ANN sobre tokens + re-score exacto, o MUVERA |
| Fortaleza | términos exactos, identificadores, explicabilidad | semántica, paráfrasis, multilingüe | precisión fina sin cross-encoder, generalización fuera de dominio |
| Debilidad | *vocabulary mismatch* | pierde detalle, términos raros | **almacenamiento** y complejidad de infraestructura |
| Analogía | `grep` con pesos | resumen en un punto del espacio | comparar **palabra por palabra** y quedarse con la mejor pareja |

> 💡 **Analogía con recsys.** BM25 ≈ filtrado por *features* explícitas; denso ≈ *two-tower* de un vector por ítem; late interaction ≈ modelos de *interacción de bajo nivel* (el score se calcula sobre pares de componentes) que son más ricos que el producto punto único pero caros de indexar. El patrón recomendador "retrieval barato → ranker caro" (recsys-course/06–07) es exactamente el de hoy: *recall* con sparse+denso, *precisión* con late interaction y cross-encoder.

> ### 🎨 GEMINI-IMAGEN · ID: `M13-IMG-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **Tres paradigmas de representación para retrieval**
>
> **Qué muestra:** Una misma consulta y un mismo documento representados de tres formas: sparse (bolsa de términos con pesos), denso (un vector) y late interaction (un vector por token con MaxSim).
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Tres columnas que muestran la misma consulta como bolsa de términos, un vector y un vector por token
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, formato 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif legible. Sin ilustraciones decorativas ni texto inventado.
> 
> Tres columnas iguales con títulos arriba. Encima de las tres, en una barra gris: consulta "cómo reducir la memoria de los embeddings" y debajo documento "Cuantizar a int8 reduce 4 veces la memoria de un índice".
> Columna 1 (naranja) "Sparse (BM25 / SPLADE)": una fila de cajitas etiquetadas con términos y pesos: "memoria 2.1", "embeddings 1.7", "int8 1.4", "cuantizar 0.9", "índice 0.8"; el resto de la fila, muchas cajitas vacías con la nota "≈ 100 000 dimensiones, casi todas 0". Subtítulo: "índice invertido: término → documentos".
> Columna 2 (azul) "Denso (1 vector)": una única barra horizontal de 8 bloques de intensidades distintas, con la etiqueta "d = 1024 números" y una flecha circular con "coseno". Subtítulo: "ANN sobre un vector por documento".
> Columna 3 (verde) "Late interaction (1 vector por token)": dos pequeñas pilas de barras: arriba 4 barras para los tokens de la consulta, abajo 6 barras para los tokens del documento; líneas finas verdes desde cada token de la consulta hasta su mejor token del documento y la nota "MaxSim: suma de los mejores emparejamientos". Subtítulo: "más precisión, más almacenamiento".
> Pie de figura gris: "lexical ← — — → semántico".
> 
> Evita: texto ilegible, más de cinco términos en la columna 1, flechas cruzadas confusas, logos, iconos decorativos.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M13-IMG-01.png` en la carpeta del módulo y reemplaza esta celda por: `![Tres columnas que muestran la misma consulta como bolsa de términos, un vector y un vector por token](assets/M13-IMG-01.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> SPARSE            |  DENSO               |  LATE INTERACTION
> memoria 2.1       |  [■▪■▫▪■▫▪] d=1024   |  q: [t1][t2][t3][t4]
> embeddings 1.7    |  coseno(q,d)         |        ↘  ↓  ↙  ↓   (mejor token de d)
> int8 1.4          |                      |  d: [u1][u2][u3][u4][u5][u6]
> ...(≈100k dims)   |                      |  score = Σ_i max_j q_i·d_j
> ```

> 👀 **Qué debes observar.** De izquierda a derecha el score deja de ser "un número por documento" y pasa a depender de **más componentes**: términos con pesos → un punto → una nube de puntos. Cada paso gana expresividad y paga en almacenamiento. Ningún paradigma domina: los sistemas buenos **combinan** al menos dos.

## 2 · BM25 desde cero (y por qué sigue ganando partidos)

$$\text{BM25}(q,d)=\sum_{t\in q}\underbrace{\ln\!\Big(1+\frac{N-n_t+0.5}{n_t+0.5}\Big)}_{\text{IDF (variante de Lucene)}}\;\cdot\;\frac{f_{t,d}\,(k_1+1)}{f_{t,d}+k_1\big(1-b+b\,\tfrac{|d|}{\overline{|d|}}\big)}$$

- $N$: nº de documentos; $n_t$: documentos que contienen $t$; $f_{t,d}$: frecuencia de $t$ en $d$; $|d|$: longitud (en términos); $\overline{|d|}$: longitud media.
- $k_1\approx1.2$ controla la **saturación** de la frecuencia (con $k_1\to0$ solo importa presencia/ausencia; con $k_1\to\infty$ se vuelve lineal como TF).
- $b\in[0,1]$ controla la **normalización por longitud** ($b=0$ la ignora; $b=1$ normaliza del todo).
- Variantes del IDF: Lucene usa $\ln(1+\cdot)$ (siempre ≥ 0); la clásica Okapi usa $\ln\frac{N-n_t+0.5}{n_t+0.5}$ que puede ser **negativa** para términos en más de la mitad de los documentos (`rank_bm25` la sustituye por un suelo $\epsilon\cdot\overline{\text{IDF}}$).

**Implementación.** Como el peso de cada par (documento, término) no depende de la query, lo **precomputamos** en una matriz dispersa $W\in\mathbb{R}^{N\times V}$; buscar es multiplicar $W$ por el vector de conteos de la query. Esto es, en esencia, lo que hace un índice invertido con *impact scores*.

In [ ]:
import re, unicodedata
import scipy.sparse as sp
from collections import Counter

SUFIJOS = ("aciones", "acion", "amente", "mente", "ando", "iendo", "ing", "ed", "es", "s", "ar", "er", "ir")

def quita_acentos(t):
    n = unicodedata.normalize("NFD", t)
    return "".join(c for c in n if unicodedata.category(c) != "Mn")

def stem_ligero(t):
    """Stemmer de juguete (sufijos ES/EN). En producción: Snowball/Porter u otro analyzer del motor."""
    for s in SUFIJOS:
        if len(t) > len(s) + 3 and t.endswith(s):
            return t[:-len(s)]
    return t

def toks(texto, acentos=True, stem=False):
    t = texto.lower()
    if acentos:
        t = quita_acentos(t)
    out = re.findall(r"\w+", t)
    return [stem_ligero(x) for x in out] if stem else out

class BM25:
    """BM25 con matriz de pesos precomputada. idf='lucene' o 'okapi' (como rank_bm25.BM25Okapi, epsilon=0.25)."""
    def __init__(self, k1=1.2, b=0.75, idf="lucene"):
        self.k1, self.b, self.idf_kind = k1, b, idf
    def fit(self, docs_tokens):
        self.vocab, filas, cols, vals = {}, [], [], []
        for i, tk in enumerate(docs_tokens):
            for t, c in Counter(tk).items():
                filas.append(i); cols.append(self.vocab.setdefault(t, len(self.vocab))); vals.append(c)
        N = len(docs_tokens)
        tf = sp.csr_matrix((vals, (filas, cols)), shape=(N, len(self.vocab)), dtype=np.float64)
        dl = np.array([len(t) for t in docs_tokens], dtype=float)
        avgdl = dl.mean()
        df = np.asarray((tf > 0).sum(0)).ravel()
        if self.idf_kind == "lucene":
            idf = np.log(1 + (N - df + 0.5) / (df + 0.5))
        else:
            idf = np.log(N - df + 0.5) - np.log(df + 0.5)
            idf = np.where(idf < 0, 0.25 * idf.mean(), idf)
        fila_de = np.repeat(np.arange(N), np.diff(tf.indptr))
        w = tf.data * (self.k1 + 1) / (tf.data + self.k1 * (1 - self.b + self.b * dl[fila_de] / avgdl))
        self.W = sp.csr_matrix((w * idf[tf.indices], tf.indices, tf.indptr), shape=tf.shape).tocsc()
        return self
    def scores(self, q_tokens):
        q = np.zeros(len(self.vocab))
        for t in q_tokens:
            if t in self.vocab:
                q[self.vocab[t]] += 1                 # los términos repetidos de la query suman varias veces
        return self.W @ q

def bm25_matriz(b, k1=1.2, bb=0.75, **analyzer):
    modelo = BM25(k1, bb).fit([toks(t, **analyzer) for t in b.txt])
    return np.stack([modelo.scores(toks(q, **analyzer)) for q in b.qtxt])

# Verificación contra rank_bm25 (misma variante de IDF y mismos tokens)
from rank_bm25 import BM25Okapi
docs_t = [toks(t) for t in B1.txt]
ref = BM25Okapi(docs_t, k1=1.2, b=0.75)
mio = BM25(idf="okapi").fit(docs_t)
q0 = toks(B1.qtxt[0])
print("coincide con rank_bm25.BM25Okapi:", bool(np.allclose(ref.get_scores(q0), mio.scores(q0))))
for b in BENCHES:
    b.S["bm25"] = bm25_matriz(b)
display(B1.resumen())

> 👀 **Qué debes observar.** Las puntuaciones coinciden con `rank_bm25` hasta el redondeo (`True`): tu BM25 es correcto. Compara el nDCG@10 del banco canónico con el de paráfrasis (siguiente celda): BM25 es muy fuerte cuando query y documento comparten vocabulario y se desploma cuando no.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(14, 3.6))
tf_ = np.arange(0, 21)
for k1 in (0.0, 0.6, 1.2, 2.0, 5.0):
    axs[0].plot(tf_, tf_ * (k1 + 1) / (tf_ + k1 + 1e-9), label=f"k1={k1}")
axs[0].set_title("Saturación de la frecuencia (b=0)"); axs[0].set_xlabel("frecuencia del término f"); axs[0].set_ylabel("peso del término"); axs[0].legend(fontsize=8)
dl = np.linspace(0.2, 3, 50)
for bb in (0.0, 0.5, 0.75, 1.0):
    axs[1].plot(dl, 3 * 2.2 / (3 + 1.2 * (1 - bb + bb * dl)), label=f"b={bb}")
axs[1].set_title("Normalización por longitud (f=3, k1=1.2)"); axs[1].set_xlabel("|d| / longitud media"); axs[1].set_ylabel("peso del término"); axs[1].legend(fontsize=8)
nombres, valores = [], []
for nombre, kw in [("sin tildes", dict(acentos=True)), ("con tildes", dict(acentos=False)), ("sin tildes + stem", dict(acentos=True, stem=True))]:
    nombres.append(nombre); valores.append([b.ndcg(bm25_matriz(b, **kw)) for b in BENCHES])
x = np.arange(len(nombres)); w = 0.38
axs[2].bar(x - w / 2, [v[0] for v in valores], w, label=B1.nombre[:10], color="#1F6FEB")
axs[2].bar(x + w / 2, [v[1] for v in valores], w, label="paráfrasis", color="#F28E2B")
axs[2].set_xticks(x); axs[2].set_xticklabels(nombres, fontsize=8); axs[2].set_ylabel("nDCG@10"); axs[2].set_title("Efecto del analyzer"); axs[2].legend(fontsize=8)
plt.tight_layout(); plt.show()
display(pd.DataFrame(valores, index=nombres, columns=["canónico", "paráfrasis"]).round(3))

> 👀 **Qué debes observar.** (1) A la izquierda, con $k_1$ alto la curva casi no satura (comportamiento tipo TF); con $k_1$ bajo, repetir un término apenas suma. (2) En el centro, $b=1$ castiga fuertemente a los documentos largos. (3) A la derecha, el *analyzer* mueve las métricas: quitar tildes y hacer stemming acercan variantes de una palabra. El efecto es distinto en cada banco: **ajusta el analyzer a tu idioma** (módulos 01–05). BM25 en el banco de paráfrasis rinde mucho peor porque casi no hay palabras en común (y cero entre idiomas).

## 3 · Sparse aprendido: SPLADE

BM25 solo puede dar peso a palabras que **están** en el texto. **SPLADE** (Formal et al., 2021) usa el cabezal de **lenguaje enmascarado (MLM)** de un transformer: para cada token del texto el modelo predice una distribución sobre **todo el vocabulario** $V$ (≈ 30 000–250 000 términos), de modo que el texto "activa" también sinónimos y términos relacionados (**expansión**) y re-pondera los términos originales.

Sea $z_{t,j}$ el logit MLM del token $t$ para el término $j\in V$. El peso del término $j$ en el texto es

$$w_j=\max_{t}\;\log\!\big(1+\operatorname{ReLU}(z_{t,j})\big)$$

(SPLADE v1 sumaba sobre $t$; **v2 usa max-pooling**, más estable). El score es el producto punto de dos vectores dispersos: $s(q,d)=\sum_j w^q_j\,w^d_j$, calculable con un **índice invertido** igual que BM25.

**Esparcidad por regularización.** Sin control, el MLM activaría miles de términos. Se entrena con una pérdida de ranking (contrastiva y, en SPLADE-v3, KL-Div + MarginMSE) más el regularizador **FLOPS**:
$$\mathcal{L}_{FLOPS}=\sum_{j\in V}\bar a_j^{\,2},\qquad \bar a_j=\frac1{B}\sum_{b=1}^{B}w^{(b)}_j,$$
que penaliza los términos que se activan **en muchos documentos** (porque ese es el coste del índice invertido: listas de postings largas). Se aplican pesos $\lambda_q,\lambda_d$ distintos a queries y documentos: así se pueden tener queries muy cortas y documentos más ricos.

> 💡 **Por qué importa.** SPLADE conserva la **explicabilidad** y el índice invertido de BM25 pero resuelve parcialmente el *vocabulary mismatch*. Existe también la variante **inference-free**: la query se pondera con una tabla (sin pasar por el modelo) y solo el documento se codifica con transformer; reduce la latencia de la query (trabajo de OpenSearch, Geng et al. 2024).

In [ ]:
def splade_pool(logits, mask):
    """logits (B,T,V) del cabezal MLM, mask (B,T). w_j = max_t log(1 + ReLU(z_tj)) -> (B,V)."""
    act = np.log1p(np.maximum(logits, 0.0)) * mask[..., None]
    return act.max(1)

def flops_reg(W):
    """Regularizador FLOPS sobre un lote de vectores sparse W (B,V): suma_j (media_b W_bj)^2."""
    return float((W.mean(0) ** 2).sum())

# ⚠️ LOGITS INVENTADOS A MANO para ilustrar la expansión (NO son salida de un modelo real).
VOC = ["gato", "felino", "mascota", "animal", "perro", "coche", "ruta", "casa"]
z_doc1 = np.array([[[4.0, 2.5, 2.0, 1.2, 0.8, -1.0, -2.0, -1.5],      # token "gato"
                    [-1.0, -0.5, 0.2, 0.9, 0.1, -1.0, -2.0, 0.6]]])    # token "duerme"
z_doc2 = np.array([[[-2.0, -2.0, -1.0, -0.5, -1.0, 3.5, 2.0, 0.3],     # token "coche"
                    [-1.5, -1.0, -1.0, -0.5, -1.0, 1.0, 3.0, 0.2]]])   # token "ruta"
m2 = np.ones((1, 2))
W1, W2 = splade_pool(z_doc1, m2)[0], splade_pool(z_doc2, m2)[0]
print("pesos doc1:", dict(zip(VOC, np.round(W1, 2))))
print("score(doc1, doc2) =", round(float(W1 @ W2), 3), "· FLOPS del par =", round(flops_reg(np.stack([W1, W2])), 3))

fig, axs = plt.subplots(1, 2, figsize=(11, 3.2), sharey=True)
for ax, W, tit, col in ((axs[0], W1, 'Documento: "el gato duerme"', "#1F6FEB"), (axs[1], W2, 'Documento: "el coche por la ruta"', "#F28E2B")):
    ax.bar(VOC, W, color=col); ax.set_title(tit, fontsize=10); ax.tick_params(axis="x", rotation=45)
axs[0].set_ylabel("peso SPLADE (ilustrativo)")
plt.suptitle("Expansión SPLADE: el texto activa términos que NO contiene (datos inventados)", fontsize=10)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** El primer documento no contiene "felino" ni "mascota" pero los **activa** con peso alto (expansión), y el segundo activa términos de otro dominio: su score cruzado es ~0. El regularizador FLOPS premia que los dos documentos usen **términos distintos** (listas de postings cortas). Estos números son ilustrativos; la siguiente celda lo repite con un modelo real si está disponible.

> ### 🎨 GEMINI-IMAGEN · ID: `M13-IMG-02` · ⚠️ REEMPLAZAR ESTA CELDA
> **Cómo SPLADE expande un texto con el cabezal MLM**
>
> **Qué muestra:** El flujo texto → transformer → logits MLM sobre todo el vocabulario → ReLU/log/max → vector disperso con términos expandidos.
> **Formato:** PNG 16:9, 1920x1080 | **Texto alternativo:** Flujo de SPLADE desde el texto hasta un vector disperso con términos expandidos
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Diagrama técnico limpio, fondo blanco, formato 16:9. Paleta: azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280. Tipografía sans-serif. Sin ilustraciones decorativas ni texto inventado.
> 
> De IZQUIERDA a DERECHA, cuatro bloques conectados por flechas grises:
> 1) Bloque gris "Texto": la frase "el gato duerme" con tres tokens en cajitas: "el", "gato", "duerme".
> 2) Bloque azul "Transformer + cabezal MLM": una caja con la etiqueta "logits z(t, j) para TODO el vocabulario" y debajo una matriz de 3 filas (tokens) por 8 columnas (palabras: gato, felino, mascota, animal, perro, coche, ruta, casa) con celdas más oscuras donde el valor es alto (gato-gato oscuro, gato-felino medio, gato-mascota medio).
> 3) Bloque naranja "ReLU → log(1 + x) → máximo sobre los tokens": una flecha que colapsa las 3 filas en una sola fila de 8 valores.
> 4) Bloque verde "Vector disperso": una fila de 8 barras verticales con alturas aproximadas gato 1.61, felino 1.25, mascota 1.10, animal 0.79, perro 0.59, coche 0, ruta 0, casa 0.47; las barras de felino, mascota, animal y perro llevan una etiqueta "expansión" (no están en el texto original) y las de coche y ruta están vacías.
> Debajo del bloque 4, una nota gris: "se guarda en un índice invertido: término → documentos".
> 
> Evita: texto ilegible, más columnas de las indicadas, flechas cruzadas, ilustraciones de gatos, logos, valores distintos de los dados.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M13-IMG-02.png` en la carpeta del módulo y reemplaza esta celda por: `![Flujo de SPLADE desde el texto hasta un vector disperso con términos expandidos](assets/M13-IMG-02.png)`
>
> **Mientras tanto (placeholder):**
> ```text
> "el gato duerme" -> [Transformer+MLM] -> z(t,j) (3 x |V|)
>       -> ReLU, log(1+x), max_t ->  gato 1.61 | felino 1.25* | mascota 1.10* | animal 0.79* | perro 0.59* | casa 0.47
>                                    (* = términos expandidos que no estaban en el texto)
> ```

> 👀 **Qué debes observar.** La expansión ocurre **antes** del índice: el costo en tiempo de consulta es el de un índice invertido normal, pero el documento ya "contiene" sinónimos. El precio es el tamaño del índice (más términos por documento) y la necesidad de ajustar la esparcidad.

### Con un modelo real
Primero **desde cero**: aplicamos la fórmula al cabezal MLM de un modelo *sin* entrenar para ranking (BERT multilingüe destilado) para ver por qué hace falta el entrenamiento y el regularizador: el número de términos activos es enorme. Después usamos un **sparse aprendido** con `sentence-transformers` (`SparseEncoder`).

In [ ]:
import torch
SPLADE_BASE = "distilbert/distilbert-base-multilingual-cased"          # MLM crudo (NO entrenado para ranking)
SPARSE_MODELOS = ["opensearch-project/opensearch-neural-sparse-encoding-multilingual-v1", "naver/splade-v3"]

def splade_mlm_encode(tok, mlm, textos, max_len=128):
    """SPLADE desde cero con transformers: logits MLM -> log(1+ReLU) -> max sobre tokens. Devuelve (B,V) denso."""
    lote = tok(textos, padding=True, truncation=True, max_length=max_len, return_tensors="pt").to(device)
    with torch.no_grad():
        z = mlm(**lote).logits                                     # (B,T,V)
    act = torch.log1p(torch.relu(z)) * lote["attention_mask"].unsqueeze(-1)
    return act.max(1).values.cpu().numpy()

def terminos_top(tok, w, k=8):
    ids = np.argsort(-w)[:k]
    return [(tok.convert_ids_to_tokens(int(i)), round(float(w[i]), 2)) for i in ids if w[i] > 0]

try:
    from transformers import AutoTokenizer, AutoModelForMaskedLM
    tok_s, mlm_s = AutoTokenizer.from_pretrained(SPLADE_BASE), AutoModelForMaskedLM.from_pretrained(SPLADE_BASE).to(device).eval()
    frase = ["El gato duerme en el sofá"]
    W = splade_mlm_encode(tok_s, mlm_s, frase)[0]
    print("MLM crudo · términos activos (>0):", int((W > 0).sum()), "de", len(W))
    print("top:", terminos_top(tok_s, W))
except Exception as e:
    tok_s = mlm_s = None
    print("MLM no disponible:", type(e).__name__, str(e)[:90])

In [ ]:
def carga_sparse():
    """Sparse aprendido con sentence-transformers.SparseEncoder; (nombre, modelo) o (None, None)."""
    try:
        from sentence_transformers import SparseEncoder
    except ImportError:
        return None, None
    for nombre in SPARSE_MODELOS:
        try:
            return nombre, SparseEncoder(nombre, device=device)
        except Exception as e:
            print(f"[sparse] {nombre}: no disponible ({type(e).__name__}: {str(e)[:60]})")
    return None, None

sparse_nombre, sparse_enc = carga_sparse()
if sparse_enc is not None:
    for b in BENCHES:
        sim = sparse_enc.similarity(sparse_enc.encode_query(b.qtxt), sparse_enc.encode_document(b.txt))
        b.S["sparse-aprendido"] = sim.cpu().numpy() if hasattr(sim, "cpu") else np.asarray(sim)
    d0 = sparse_enc.encode_document(B1.txt[:1])
    print(f"sparse aprendido = {sparse_nombre}")
    print("top términos del primer documento:", [(t, round(float(w), 2)) for t, w in sparse_enc.decode(d0[0], top_k=8)])
    display(B2.resumen())
else:
    print("No hay un modelo sparse aprendido accesible (sin red o modelo restringido): seguimos con BM25 + denso + late interaction.")

> 👀 **Qué debes observar.** (1) El MLM crudo activa **cientos o miles** de términos por frase (la mayoría con peso pequeño): sin entrenamiento y sin FLOPS no hay esparcidad; un índice invertido con eso sería inservible. (2) El modelo sparse entrenado (si lo ves) conserva decenas de términos y mezcla los de la frase con expansiones. (3) En el banco de **paráfrasis** el sparse aprendido debería superar a BM25, pero **no** reemplaza a un buen denso multilingüe en consultas entre idiomas (depende del checkpoint: los modelos de SPLADE originales son monolingües en inglés).

## 4 · Late interaction: ColBERT y MaxSim desde cero

**Idea (Khattab & Zaharia, 2020).** El bi-encoder denso comprime el texto en un punto. ColBERT guarda **un vector por token** (normalizado, $d=128$ tras una proyección lineal sobre BERT) y calcula la relevancia con una **interacción tardía** barata (sin pasar los dos textos juntos por un transformer):

$$S(q,d)=\sum_{i=1}^{|q|}\;\max_{j=1}^{|d|}\;\langle \hat{q}_i,\hat{d}_j\rangle$$

Cada token de la query "busca" su mejor pareja en el documento y se suman las similitudes. Sigue siendo **precomputable** para los documentos (como un bi-encoder) pero conserva información fina por token. Detalles del paper original: marcadores `[Q]`/`[D]`, *query augmentation* (se rellena la query con `[MASK]` hasta una longitud fija, que actúa como expansión suave) y filtrado de puntuación en el documento.

**ColBERTv2 (Santhanam et al., 2021)** ataca el almacenamiento con **compresión residual**: guarda el id del **centroide** más cercano y el residuo $r=v-c$ cuantizado a 1–2 bits por dimensión (el paper informa reducciones del espacio de 6–10×), además de entrenar con supervisión destilada de un cross-encoder y *hard negatives*. **PLAID (2022)** acelera la búsqueda con **interacción por centroides** y poda de candidatos.

**Coste:** un documento de 200 tokens con $d=128$ en `float16` ocupa $200\times128\times2=51.2$ KB (¡frente a 2 KB de un vector denso de 1024 en `float16`!). Este es el límite práctico de la técnica.

In [ ]:
import zlib
_BANCO_NGRAMAS = {}

def vec_ngrama(g, d):
    v = _BANCO_NGRAMAS.get((g, d))
    if v is None:
        v = np.random.default_rng(zlib.crc32(g.encode())).normal(size=d).astype(np.float32)
        _BANCO_NGRAMAS[(g, d)] = v
    return v

def vec_token(tok, d=64):
    """Embedding de JUGUETE de un token: suma de vectores aleatorios deterministas de sus 3-gramas de caracteres.
    Palabras que comparten letras (indexación/indexar) quedan cerca. No es semántico, pero hace los ejemplos reproducibles."""
    w = f"<{tok}>"
    grams = [w[i:i + 3] for i in range(len(w) - 2)] or [w]
    v = sum(vec_ngrama(g, d) for g in grams)
    return v / np.linalg.norm(v)

def multi_juguete(textos, d=64, max_tok=48):
    out = []
    for t in textos:
        tk = toks(t)[:max_tok] or ["vacio"]
        out.append(np.stack([vec_token(x, d) for x in tk]).astype(np.float32))
    return out

def maxsim(Q, D):
    """Q (nq,d), D (nd,d), filas normalizadas. S = suma_i max_j <q_i, d_j>."""
    return float((Q @ D.T).max(axis=1).sum())

def maxsim_matriz(QV, DV, trozo=32):
    """Todos los pares (query, doc) sin bucles de Python por documento: concatena los tokens de los documentos,
    una sola multiplicación de matrices por trozo de queries y máximos por segmento con np.maximum.reduceat."""
    Dcat = np.concatenate(DV)
    ini_d = np.cumsum([0] + [len(x) for x in DV[:-1]])
    out = np.empty((len(QV), len(DV)), np.float32)
    for s in range(0, len(QV), trozo):
        qs = QV[s:s + trozo]
        Qcat = np.concatenate(qs)
        ini_q = np.cumsum([0] + [len(x) for x in qs[:-1]])
        mx = np.maximum.reduceat(Qcat @ Dcat.T, ini_d, axis=1)          # (Tq, nd): mejor token de cada doc
        out[s:s + len(qs)] = np.add.reduceat(mx, ini_q, axis=0)          # suma sobre los tokens de cada query
    return out

t0 = time.perf_counter()
for b in BENCHES:
    b.DV, b.QV = multi_juguete(b.txt), multi_juguete(b.qtxt)
    b.S["colbert-juguete"] = maxsim_matriz(b.QV, b.DV)
print(f"MaxSim de {len(B1.qs)}x{len(B1.ids)} pares en {time.perf_counter()-t0:.2f}s")
print("coincide con la versión ingenua:", bool(np.isclose(B1.S["colbert-juguete"][3, 7], maxsim(B1.QV[3], B1.DV[7]), atol=1e-4)))
display(pd.concat({b.nombre[:12]: b.resumen() for b in BENCHES}))

> 👀 **Qué debes observar.** La versión vectorizada coincide con la fórmula ingenua. El "ColBERT de juguete" usa embeddings de tokens por 3-gramas de caracteres (léxicos, no semánticos): se comporta como un **BM25 difuso** (hace match de variantes ortográficas) y sirve para entender el *mecanismo*. Con embeddings contextuales entrenados (ColBERT real, BGE-M3 más abajo) cada token ya incorpora el contexto y la sinonimia.

> ### 🎬 GEMINI-VIDEO · ID: `M13-VID-01` · ⚠️ REEMPLAZAR ESTA CELDA
> **MaxSim token a token**
>
> **Qué muestra:** Animación de cómo cada token de la consulta busca su mejor pareja entre los tokens del documento y cómo se suman los máximos (ejemplo con números exactos).
> **Formato:** MP4 16:9, 1080p | Duración: 30 s | **Texto alternativo:** Animación de la matriz de similitudes 3x4 con los máximos por fila sumados en un score de 2.51
>
> **Prompt para Gemini (copiar y pegar):**
> ```text
> Video explicativo 16:9, fondo blanco, estilo diagrama técnico limpio, paleta azul #1F6FEB, naranja #F28E2B, verde #2CA02C, gris #6B7280, tipografía sans-serif, ritmo calmado, SIN voz en off ni música; solo texto en pantalla en español (términos técnicos en inglés).
> 
> Datos EXACTOS (no inventar otros). Tokens de la consulta (columna izquierda, azul): "buscar", "vecinos", "red". Tokens del documento (fila superior, gris): "grafo", "capas", "vecinos", "búsqueda". Matriz de similitudes (filas = tokens de la consulta, columnas = tokens del documento):
> buscar  : grafo 0.12, capas 0.08, vecinos 0.35, búsqueda 0.82
> vecinos : grafo 0.20, capas 0.10, vecinos 0.91, búsqueda 0.30
> red     : grafo 0.78, capas 0.25, vecinos 0.22, búsqueda 0.15
> 
> Escena 1 (0–5 s): título "ColBERT: late interaction con MaxSim". Aparecen a la izquierda los 3 tokens de la consulta como cajitas azules y arriba los 4 tokens del documento como cajitas grises.
> Escena 2 (5–12 s): se rellena la matriz 3×4 con los 12 números, celda a celda, con un degradado de color según el valor.
> Escena 3 (12–20 s): fila por fila, se resalta en verde el máximo de cada fila ("buscar → búsqueda 0.82", "vecinos → vecinos 0.91", "red → grafo 0.78") y se dibuja una línea verde entre el token de la consulta y su mejor pareja; los demás valores se atenúan.
> Escena 4 (20–26 s): los tres máximos viajan a una suma en el lado derecho: "0.82 + 0.91 + 0.78 = 2.51" con el rótulo "score MaxSim = 2.51".
> Escena 5 (26–30 s): texto final "Cada token de la consulta elige su mejor pareja; el documento no se comprime en un solo vector".
> 
> Evita: voz, música, números distintos de los dados, texto ilegible, elementos decorativos, cambios bruscos de cámara.
> ```
>
> **Cómo insertarlo:** guarda el archivo como `assets/M13-VID-01.mp4` en la carpeta del módulo y reemplaza esta celda por: `<video src="assets/M13-VID-01.mp4" controls width="100%"></video>`
>
> **Mientras tanto (placeholder):**
> ```text
>               grafo  capas  vecinos  búsqueda
> buscar         0.12   0.08   0.35    [0.82]
> vecinos        0.20   0.10  [0.91]    0.30
> red           [0.78]  0.25   0.22     0.15
> MaxSim = 0.82 + 0.91 + 0.78 = 2.51
> ```

> 👀 **Qué debes observar.** El score no es un coseno único sino la **suma de tres emparejamientos independientes**: cada token de la consulta elige su mejor pareja, aunque esté en otra posición del documento. Por eso late interaction tolera bien reordenamientos y paráfrasis parciales, y por eso "explica" qué parte del documento justificó el score.

In [ ]:
def heatmap_maxsim(q_toks, Q, d_toks, D, titulo):
    S = Q @ D.T
    fig, ax = plt.subplots(figsize=(0.6 * len(d_toks) + 3, 0.5 * len(q_toks) + 1.6))
    im = ax.imshow(S, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(d_toks))); ax.set_xticklabels(d_toks, rotation=60, ha="right", fontsize=8)
    ax.set_yticks(range(len(q_toks))); ax.set_yticklabels(q_toks, fontsize=8)
    for i, j in enumerate(S.argmax(1)):
        ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="#F28E2B", lw=2.2))
    ax.set_title(f"{titulo}\nMaxSim = {S.max(1).sum():.2f}", fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.03); plt.tight_layout(); plt.show()

par = {k[0]: k for k in PARAFRASIS}["hnsw"]
q_t, d_t = toks(par[5]), toks(par[2])                       # query en español y documento en español
heatmap_maxsim(q_t, multi_juguete([par[5]])[0], d_t[:24], multi_juguete([par[2]])[0][:24], "Matriz token-token (embeddings de juguete)")

> 👀 **Qué debes observar.** Los recuadros naranjas son los máximos por fila (el `max` de MaxSim). Con vectores de juguete solo hay similitud alta entre palabras con letras en común ("vectores"/"vecinos", "grafo"); con ColBERT entrenado la fila "buscar" apuntaría a "búsqueda" o "devolviendo" por **significado**. Observa también que las palabras vacías (artículos, preposiciones) también aportan; ColBERT original filtra puntuación pero no stopwords.

### Coste de almacenamiento y compresión residual (ColBERTv2, versión mini)
Aprendemos $K$ centroides con k-means sobre los vectores de token; guardamos por token el **id del centroide** (1 byte para $K\le256$) y el **residuo** $v-c$ cuantizado a $n$ bits por dimensión (con umbrales por cuantiles; reconstruimos con la media de cada cubeta).

In [ ]:
from sklearn.cluster import KMeans

def comprime_residual(V, C, nbits):
    """ColBERTv2 en pequeño: (id de centroide, residuo cuantizado a nbits por dimensión). Devuelve datos para reconstruir."""
    a = (-2 * V @ C.T + (C ** 2).sum(1)).argmin(1)
    R = V - C[a]
    umbrales = np.quantile(R, np.linspace(0, 1, 2 ** nbits + 1)[1:-1]) if nbits > 1 else np.array([0.0])
    codigos = np.digitize(R, umbrales)
    valores = np.array([R[codigos == k].mean() if (codigos == k).any() else 0.0 for k in range(2 ** nbits)])
    return a, codigos.astype(np.uint8), valores

def descomprime(a, codigos, valores, C):
    V = C[a] + valores[codigos]
    return (V / np.linalg.norm(V, axis=1, keepdims=True)).astype(np.float32)

b = B1
Vtodos = np.concatenate(b.DV)
longitudes = np.array([len(x) for x in b.DV]); cortes = np.cumsum(np.r_[0, longitudes])
C = KMeans(256, n_init=1, random_state=SEED).fit(Vtodos).cluster_centers_.astype(np.float32)
dim = Vtodos.shape[1]
filas = [("float32", 4 * dim, b.ndcg(b.S["colbert-juguete"]))]
filas.append(("float16", 2 * dim, b.ndcg(maxsim_matriz(b.QV, [x.astype(np.float16).astype(np.float32) for x in b.DV]))))
for nb_ in (3, 2, 1):
    a_, cod_, val_ = comprime_residual(Vtodos, C, nb_)
    Vh = descomprime(a_, cod_, val_, C)
    DVh = [Vh[cortes[i]:cortes[i + 1]] for i in range(len(b.DV))]
    filas.append((f"centroide + residuo {nb_} bit/dim", 1 + dim * nb_ / 8, b.ndcg(maxsim_matriz(b.QV, DVh))))
tabla_c = pd.DataFrame(filas, columns=["representación", "bytes por token", "nDCG@10"])
display(tabla_c.round(3))
fig, ax = plt.subplots(figsize=(6.6, 3.5))
ax.plot(tabla_c["bytes por token"], tabla_c["nDCG@10"], "o-", color="#2CA02C")
for r in tabla_c.itertuples():
    ax.annotate(r.representación, (r._2, r._3), textcoords="offset points", xytext=(4, 6), fontsize=7)
ax.set_xscale("log"); ax.set_xlabel("bytes por token (log)"); ax.set_ylabel("nDCG@10"); ax.set_title("Compresión residual: espacio vs calidad"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

> 👀 **Qué debes observar.** Bajar de 256 B (`float32`, $d=64$) a ≈ 9–25 B por token con residuos de 1–3 bits cuesta muy poco nDCG: **los residuos tienen poca energía** una vez que restas el centroide. Esa es la idea central de ColBERTv2. Con $d=128$ y 2 bits son 33 B por token frente a 256 B en `float16`. Atención: este es un experimento a escala diminuta con embeddings de juguete; la reducción de 6–10× es la cifra que **reporta el paper** para ColBERTv2 real.

### PLAID en pequeño: interacción por centroides
Buscar con MaxSim exacto sobre toda la colección es $O(\text{tokens})$. PLAID (Santhanam et al., 2022) usa los centroides como **proxy**: cada documento se representa por el *conjunto de centroides* de sus tokens; la similitud query-centroide ($q_i\cdot c_k$) se calcula **una sola vez por query** y el score aproximado de un documento es $\tilde S(q,d)=\sum_i\max_{k\in\text{cent}(d)} q_i\cdot c_k$ (un MaxSim con vectores "cuantizados" a su centroide). Se conservan los `ncand` mejores y solo a esos se les calcula el MaxSim exacto.

In [ ]:
doc_de_tok = np.repeat(np.arange(len(b.DV)), longitudes)
asig = (-2 * Vtodos @ C.T + (C ** 2).sum(1)).argmin(1)
MASK_C = np.zeros((len(b.DV), len(C)), bool)
MASK_C[doc_de_tok, asig] = True                                   # doc x centroide: ¿algún token del doc cae ahí?

def interaccion_centroides(Q):
    sqc = Q @ C.T                                                 # (nq_tokens, K): una vez por query
    return np.where(MASK_C[None], sqc[:, None, :], -np.inf).max(-1).sum(0)

exacto = b.S["colbert-juguete"]
top_exacto = np.argsort(-exacto, axis=1)[:, :10]
aprox = np.stack([interaccion_centroides(Q) for Q in b.QV])
ncands = [10, 20, 50, 100, 200]
rec = []
for nc in ncands:
    top_nc = np.argsort(-aprox, axis=1)[:, :nc]
    rec.append(np.mean([len(set(top_nc[i]) & set(top_exacto[i])) / 10 for i in range(len(b.QV))]))
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot([100 * n / len(b.DV) for n in ncands], rec, "o-", color="#1F6FEB")
ax.set_xlabel("% de documentos que pasan al MaxSim exacto"); ax.set_ylabel("recall del top-10 exacto")
ax.set_title("PLAID mini: filtrar con centroides antes del MaxSim exacto"); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()
print({n: round(float(r), 3) for n, r in zip(ncands, rec)})

> 👀 **Qué debes observar.** Con ~12 % de los documentos pasando al MaxSim exacto ya se recupera la mayor parte del top-10 exacto: la interacción por centroides es un **filtro barato** (una multiplicación query×centroides y búsquedas en tabla). En PLAID real esto se combina con listas invertidas centroide→documentos y con poda de centroides de baja puntuación. Con vectores de juguete la curva es peor de lo que verías con ColBERT real (menos estructura de clusters).